# Sea Ice Carbon Budgets

# Import Required Packages and Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

%pylab inline

import numpy as np
import xarray as xr
import pandas as pd

from cartopy import crs, feature
from matplotlib import cm
from matplotlib import pyplot as plt


# Explore RECCAP2 Regional Masks

## Download Southern Ocean mask

In [ ]:
def get_southern_ocean_subregions(
    url='https://github.com/RECCAP2-ocean/RECCAP2-shared-resources/raw/master/data/regions/RECCAP2_region_masks_all_v20210412.nc',
    dest='../data/regions/'
):
    import itertools
    from pathlib import Path as posixpath
    import pooch
    
    fname = pooch.retrieve(url, None, posixpath(url).name, dest)
    ds = xr.open_dataset(fname)

    mask = ds.southern
    
    atlantic = (((mask.lon > 290) | (mask.lon <=  20)) & (mask > 0)).astype(int) * 1
    indian   = (((mask.lon >  20) & (mask.lon <= 147)) & (mask > 0)).astype(int) * 2
    pacific  = (((mask.lon > 147) & (mask.lon <= 290)) & (mask > 0)).astype(int) * 3

    mask = xr.Dataset()
    mask['biomes'] = ds.southern.copy()
    mask['basins'] = (pacific + atlantic + indian).transpose('lat', 'lon')
    
    mask['subregions'] = (mask.basins * 3 + mask.biomes - 3).where(lambda a: a>0).fillna(0).astype(int)
    
    basin = ['ATL', 'IND', 'PAC']
    biome = ['STSS', 'SPSS', 'ICE']
    names = ['-'.join(l) for l in itertools.product(basin, biome)]    
    mask['names'] = xr.DataArray(names, coords={'idx': range(1, 10)}, dims=('idx'))
    mask['names'].attrs['description'] = 'Names for the subregions'
    
    mask['subregions'].attrs['description'] = '(basins * 3 + biomes - 3)'
    mask['basins'].attrs['description'] = 'Atlantic = 1, Indian = 2, Pacific = 3'
    mask['biomes'].attrs['description'] = 'Biomes based on Fay and McKinley (2014), STSS=1, SPSS=2, ICE=3'
    mask.attrs['source'] = url
    mask.attrs['date'] = pd.Timestamp.today().strftime('%Y-%m-%d')
    return mask


In [ ]:
da = get_southern_ocean_subregions()
da


In [ ]:
da.subregions.plot()


In [ ]:
regions = get_southern_ocean_subregions().sel(lat=slice(-90, -34))
regions



In [ ]:

so = regions.subregions.where(lambda x: x!=0)


In [ ]:
for idx, val in enumerate(list(regions.names.values)):
    print(idx+1, val)


In [ ]:
names = ['Atlantic STSS',
 'Atlantic SPSS',
 'Atlantic ICE',
 'Indian STSS',
 'Indian SPSS',
 'Indian ICE',
 'Pacific STSS',
 'Pacific SPSS',
 'Pacific ICE']

clist = np.array(cm.tab20c.colors)
colors = clist[[10, 9, 8, 2, 1, 0, 6, 5, 4,]].tolist()


##  Process Southern Ocean mask

In [ ]:
fig = plt.figure(figsize=[7.5, 4], dpi=100)
ax = fig.add_subplot(111, projection=crs.SouthPolarStereo())

img = so.plot.pcolormesh(
    ax=ax, transform=crs.PlateCarree(), 
    levels=np.arange(0.5, 10), 
    cbar_kwargs=dict(pad=0.03),
    colors=colors, 
    rasterized=True,
    zorder=1)

ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=10)

y = lambda lat: [lat] * x.size
for lat in [-35, -44, -58]:
    deg_size = np.cos(np.deg2rad(lat)) ** 1.5
    n_degs = (10 / deg_size) / 2
    x = np.arange(0 + n_degs, 360 - n_degs, 1)
    
    ax.plot(x, y(lat), transform=crs.PlateCarree(), color='k', lw=0.5, ls='--', alpha=0.4, zorder=2)
    ax.text(0, lat, f"{abs(lat)}°S", ha='center', va='center', transform=crs.PlateCarree(), size=8, alpha=0.4, zorder=2)

img.colorbar.set_ticks(np.arange(1, 10))
img.colorbar.set_ticklabels(names)
img.colorbar.set_label('')
[s.set_lw(0) for s in img.colorbar.ax.spines.values()]

props = dict(ha='left', va='center', transform=crs.PlateCarree(), zorder=12, size=10, weight='light', fontfamily='helvetica')
ax.text(149, -35, '147°E', **props)
ax.text(20, -32, '20°E', **props)
ax.text(292, -26, '70°W', **props)

ax.spines['geo'].set_zorder(12)

img.colorbar.ax.tick_params('both', length=0)

# fig.savefig('../figures/Figure1_map.pdf', bbox_inches='tight', dpi=300)


## Extract and explore Sea Icea Biome and sub-regions

In [ ]:
ice_names = ['Atlantic ICE',
             'Indian ICE',
             'Pacific ICE']
so_ice_sectors = so.where(lambda x: x%3==0)


In [ ]:
clist = np.array(cm.tab20c.colors)
new_colors = clist[[8, 1, 5,]].tolist()
fig = plt.figure(figsize=[7.5, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.SouthPolarStereo())

img = so_ice_sectors.plot.pcolormesh(
    ax=ax, transform=crs.PlateCarree(), 
    levels=np.array([0.5, 3.5, 6.5, 9.5]), 
    cbar_kwargs=dict(pad=0.03),
    colors=new_colors,
    rasterized=True,
    zorder=1)

ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=10)

y = lambda lat: [lat] * x.size
for idx, lat in enumerate([-25, -35, -44, -58]):
    deg_size = np.cos(np.deg2rad(lat)) ** 1.5
    n_degs = (10 / deg_size) / 2
    x = np.arange(0 + n_degs, 360 - n_degs, 1)
    
    ax.plot(x, y(lat), transform=crs.PlateCarree(), color='k', lw=0.5, ls='--', alpha=0.4, zorder=2)
    # if idx != 0:
    ax.text(0, lat, f"{abs(lat)}°S", ha='center', va='center', transform=crs.PlateCarree(), size=8, alpha=0.4, zorder=2)

img.colorbar.set_ticks(np.array([2., 5., 8.]))
img.colorbar.set_ticklabels(ice_names)
img.colorbar.set_label('')
[s.set_lw(0) for s in img.colorbar.ax.spines.values()]

props = dict(ha='left', va='center', transform=crs.PlateCarree(), zorder=12, size=8, weight='light', fontfamily='helvetica')
ax.text(149, -35, '147°E', **props)
ax.text(20, -32, '20°E', **props)
ax.text(292, -26, '70°W', **props)

ax.spines['geo'].set_zorder(12)

img.colorbar.ax.tick_params('both', length=0)


In [ ]:
clist = np.array(cm.tab20c.colors)
new_colors = clist[[8, 1, 5,]].tolist()
fig = plt.figure(figsize=[7.5, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.SouthPolarStereo())

img = so_ice_sectors.plot.pcolormesh(
    ax=ax, transform=crs.PlateCarree(), 
    levels=np.array([0.5, 3.5, 6.5, 9.5]), 
    cbar_kwargs=dict(pad=0.03),
    colors=new_colors,
    rasterized=True,
    zorder=1)

ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=10)

y = lambda lat: [lat] * x.size
for idx, lat in enumerate([-33, -35, -46, -58]):
    deg_size = np.cos(np.deg2rad(lat)) ** 1.5
    n_degs = (10 / deg_size) / 2
    x = np.arange(0 + n_degs, 360 - n_degs, 1)
    
    if idx != 0:
        ax.plot(x, y(lat), transform=crs.PlateCarree(), color='k', lw=0.5, ls='--', alpha=0.4, zorder=2)
        ax.text(0, lat, f"{abs(lat)}°S", ha='center', va='center', transform=crs.PlateCarree(), size=8, alpha=0.4, zorder=2)
    else:
        ax.plot(x, y(lat), transform=crs.PlateCarree(), color='w', lw=0.05, ls='--', alpha=0.05, zorder=2)

img.colorbar.set_ticks(np.array([2., 5., 8.]))
img.colorbar.set_ticklabels(ice_names)
img.colorbar.set_label('')
[s.set_lw(0) for s in img.colorbar.ax.spines.values()]

props = dict(ha='left', va='center', transform=crs.PlateCarree(), zorder=12, size=8, weight='light', fontfamily='helvetica')
ax.text(149, -35, '147°E', **props)
ax.text(20, -32, '20°E', **props)
ax.text(292, -26, '70°W', **props)
ax.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))

ax.spines['geo'].set_zorder(12)

img.colorbar.ax.tick_params('both', length=0)


In [ ]:
file_path = '../data/regions/RECCAP2_region_masks_all_v20210412.nc'
ocean_regions = xr.open_dataset(file_path)
ocean_regions


In [ ]:
so_ice_mask = ocean_regions.southern.where(lambda x: x==3)
so_ice_mask


In [ ]:
so_ice_mask.plot.contourf();
np.unique(so_ice_mask)


In [ ]:
so_ice_sectors.plot.contourf();
np.unique(so_ice_sectors)


# CSIR-ML6 V2019 Data

## Load the data

In [ ]:
file_path = '../../../AIMES/data/pCO2_CSIRML6_v2019a.nc'
csirml6v2019 = xr.open_dataset(file_path)
csirml6v2019


In [ ]:
ds = csirml6v2019[['FCO2_smooth']].rename({'FCO2_smooth': 'fgco2'}).sel(time=slice('2000-01-01', '2016-12-31'))

# Convert longitude
fgco2 = ds.fgco2.assign_coords(lon=((ds.lon + 360) % 360)).copy()
fgco2 = fgco2.sortby('lon')
fgco2



## Process air-sea CO$_2$ fluxes in Sea Ice Biome

In [ ]:
so_ice_fgco2 = fgco2.where(so_ice_mask==3).sel(lat=slice(-90, -35))
so_ice_fgco2


In [ ]:
da = so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)
da.plot(robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='F$_{gco2}$ (molC m$^{-2}$ yr$^{-1}$)'))


In [ ]:
da = so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)

fig = plt.figure(figsize=[7.5, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))


da.plot(ax=ax, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='F$_{CO2}$ (molC m$^{-2}$ yr$^{-1}$)'), transform=crs.PlateCarree())

ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1);


In [ ]:
da = so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)

fig = plt.figure(figsize=[7.5, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))


da.plot.contourf(ax=ax, robust=True, cmap='RdBu_r', levels=20, cbar_kwargs=dict(label='F$_{CO2}$ (molC m$^{-2}$ yr$^{-1}$)', extend="both"), transform=crs.PlateCarree())

ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1);


In [ ]:
da = fgco2.sel(lat=slice(-90, -30)).mean('time', skipna=True, keep_attrs=True)


fig = plt.figure(figsize=[7.5, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=7))


da.plot.contourf(ax=ax, robust=True, cmap='RdBu_r', levels=30, cbar_kwargs=dict(label='F$_{CO2}$ (molC m$^{-2}$ yr$^{-1}$)', extend="both"), transform=crs.PlateCarree())

ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1);



In [ ]:
def process_airsea_flux(flux, mask=None):
    """
    Compute monthly mean air-sea CO2 fluxes and total carbon fluxes from a
    DataArray of molC/m²/yr, including climatological and time-series values.

    Parameters
    ----------
    flux : xr.DataArray
        Air-sea CO2 flux [molC/m²/yr] with dimensions (time, lat, lon)
    mask : xr.DataArray, optional

    Returns
    -------
    results : dict
        {
          'monthly_flux_mmol_day': xr.DataArray [mmol/m²/day per month, time-series],
          'climatology_flux_mmol_day': xr.DataArray [mmol/m²/day, month mean],
          'monthly_flux_TgC': xr.DataArray [Tg C/month, time-series],
          'climatology_flux_TgC': xr.DataArray [Tg C/month, month mean]
        }
    """

    # --- CONSTANTS ---
    Re = 6.371e6  # Earth's radius (m)
    mol_to_mmol_day = 1000 / 365.0
    molC_to_TgC_month = (12.011 / 1e12) / 12.0  # g/mol → Tg C, per month
    molC_to_TgC_year = (12.011 / 1e12)  # g/mol → Tg C, per year
    molC_to_PgC_month = (12.011 / 1e15) / 12.0  # g/mol → Pg C, per month
    molC_to_PgC_year = (12.011 / 1e15)  # g/mol → Pg C, per year
    flux_da = flux.copy()
    index = flux_da.time.dt.month.to_dataframe().drop_duplicates().index.strftime('%b')

    # --- STEP 1: Compute grid cell area (m²) from lat/lon ---
    lat = flux_da["lat"]
    lon = flux_da["lon"]
    
    # Ensure monotonic spacing
    dlat = np.abs(lat.diff("lat").mean().item())
    dlon = np.abs(lon.diff("lon").mean().item())
    
    # Convert to radians
    dlat_rad = np.deg2rad(dlat)
    dlon_rad = np.deg2rad(dlon)
    
    # Compute area per grid cell
    lat_rad = np.deg2rad(lat)
    area = (Re**2) * dlon_rad * (np.sin(lat_rad + dlat_rad/2) - np.sin(lat_rad - dlat_rad/2))
    area_da = xr.DataArray(np.broadcast_to(area.values[:, np.newaxis], flux_da.isel(lon=slice(0, None)).shape[1:]),
                           dims=("lat", "lon"),
                           coords={"lat": lat, "lon": lon},
                           name="area")

    # --- STEP 2: Apply regional mask ---
    if mask:
        area_da = area_da.where(mask)
        flux_da = flux_da.where(mask)

    # --- STEP 3: Convert to mmol/m²/day ---
    flux_mmol_day = flux_da * mol_to_mmol_day
    flux_mmol_day.name = 'FCO2 (mmol m-2 day-1)'
    
    flux_mmol_year = flux_da.copy()
    flux_mmol_year.name = 'FCO2 (mmol m-2 yr-1)'

    # --- STEP 4: Compute spatial means (time series) ---
    flux_mmol_m2_day = flux_mmol_day.mean(dim=["lat", "lon"])
    flux_mmol_m2_year = flux_mmol_year.mean(dim=["lat", "lon"])

    # --- STEP 5: Compute climatological monthly means ---
    clim = flux_mmol_m2_day.groupby("time.month").mean("time")
    clim_ = flux_mmol_m2_day.groupby("time.month").std("time").rename('std_dev')
    clim_flux_mmol_m2_day = xr.merge([clim, clim_]).to_dataframe()
    clim_flux_mmol_m2_day.index = index
    #
    clim = flux_mmol_m2_year.groupby("time.month").mean("time")
    clim_ = flux_mmol_m2_year.groupby("time.month").std("time").rename('std_dev')
    clim_flux_mmol_m2_year = xr.merge([clim, clim_]).to_dataframe()
    clim_flux_mmol_m2_year.index = index

    # --- STEP 6: Compute total flux (TgC/month & PgC/month) ---
    flux_TgC_global = (flux_da * area_da * molC_to_TgC_month).sum(dim=["lat", "lon"])
    flux_TgC_global.name = 'FCO2 (Tg C month-1)' 
    flux_TgC_month = flux_TgC_global.resample(time="ME").sum()
    clim = flux_TgC_month.groupby("time.month").mean("time")
    clim_ = flux_TgC_month.groupby("time.month").std("time").rename('std_dev')
    clim_flux_TgC = xr.merge([clim, clim_]).to_dataframe()
    clim_flux_TgC.index = index
    #
    flux_PgC_global = (flux_da * area_da * molC_to_PgC_month).sum(dim=["lat", "lon"])
    flux_PgC_global.name = 'FCO2 (Pg C month-1)' 
    flux_PgC_month = flux_PgC_global.resample(time="ME").sum()
    clim = flux_PgC_month.groupby("time.month").mean("time")
    clim_ = flux_PgC_month.groupby("time.month").std("time").rename('std_dev')
    clim_flux_PgC = xr.merge([clim, clim_]).to_dataframe()
    clim_flux_PgC.index = index
    
    # --- STEP 7: Compute total flux (TgC/yr & PgC/yr) ---
    flux_TgC_global = (flux_da * area_da * molC_to_TgC_year).sum(dim=["lat", "lon"])
    flux_TgC_global.name = 'FCO2 (Tg C yr-1)' 
    flux_TgC_yr_month = flux_TgC_global.resample(time="ME").sum()
    clim = flux_TgC_yr_month.groupby("time.month").mean("time")
    clim_ = flux_TgC_yr_month.groupby("time.month").std("time").rename('std_dev')
    clim_flux_TgC_yr = xr.merge([clim, clim_]).to_dataframe()
    clim_flux_TgC_yr.index = index
    #
    flux_PgC_global = (flux_da * area_da * molC_to_PgC_year).sum(dim=["lat", "lon"])
    flux_PgC_global.name = 'FCO2 (Pg C yr-1)' 
    flux_PgC_yr_month = flux_PgC_global.resample(time="ME").sum()
    clim = flux_PgC_yr_month.groupby("time.month").mean("time")
    clim_ = flux_PgC_yr_month.groupby("time.month").std("time").rename('std_dev')
    clim_flux_PgC_yr = xr.merge([clim, clim_]).to_dataframe()
    clim_flux_PgC_yr.index = index

    # --- STEP 8: Return everything neatly packaged ---
    output_dict = {"flux_mmol_m2_day": flux_mmol_m2_day,
                   "climatology_flux_mmol_m2_day": clim_flux_mmol_m2_day,
                   "flux_mmol_m2_year": flux_mmol_m2_year,
                   "climatology_flux_mmol_m2_year": clim_flux_mmol_m2_year,
                   "monthly_flux_TgC": flux_TgC_month,
                   "climatology_flux_TgC": clim_flux_TgC,
                   "monthly_flux_TgC_yr": flux_TgC_yr_month,
                   "climatology_flux_TgC_yr": clim_flux_TgC_yr,
                   "monthly_flux_PgC": flux_PgC_month,
                   "climatology_flux_PgC": clim_flux_PgC,
                   "monthly_flux_PgC_yr": flux_PgC_yr_month,
                   "climatology_flux_PgC_yr": clim_flux_PgC_yr}
    
    return output_dict


In [ ]:
dict_fgco2 = process_airsea_flux(so_ice_fgco2)
list(dict_fgco2.keys())


In [ ]:
dict_fgco2['flux_mmol_m2_day']


In [ ]:
dict_fgco2['climatology_flux_mmol_m2_day']


In [ ]:
dict_fgco2['monthly_flux_TgC']


In [ ]:
dict_fgco2['climatology_flux_TgC']


In [ ]:
# fig = plt.figure(figsize=[5, 5], dpi=300)
# ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))

# # Map
# map_da.plot(ax=ax, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='air-sea CO$_{2}$ (mmolC m$^{-2}$ day$^{-1}$)'), transform=crs.PlateCarree(), zorder=-1)
# ax.coastlines(lw=0.5, zorder=11)
# ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=1)


In [ ]:
# PLOTTING
map_da = (1000/365)*so_ice_fgco2.mean('time', skipna=True, keep_attrs=True).sel(lat=slice(-90, -35))

fig = plt.figure(figsize=[20, 5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_da.plot(ax=ax1, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='air-sea CO$_{2}$ (mmolC m$^{-2}$ day$^{-1}$)'), transform=crs.PlateCarree(), zorder=-1)
ax1.coastlines(lw=0.5, zorder=11)
ax1.add_feature(feature.LAND, facecolor='#cccccc', zorder=1)

# Time series
dict_fgco2['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='b', lw=2.5, ms=7, label='Climatology (2000-2016)')
dict_fgco2['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.5, ms=3, alpha=0.5, label='Monthly (2000-2016)')
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (TgC/month)')
ax2.legend(frameon=False)
ax3.legend(frameon=False)

fig.suptitle('Southern Ocean Sea Ice Biome', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


In [ ]:
# PLOTTING
map_da = (1000/365)*so_ice_fgco2.mean('time', skipna=True, keep_attrs=True).sel(lat=slice(-90, -35))

fig = plt.figure(figsize=[20, 5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_da.plot(ax=ax1, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='air-sea CO$_{2}$ (mmolC m$^{-2}$ day$^{-1}$)'), transform=crs.PlateCarree(), zorder=-1)
ax1.coastlines(lw=0.5, zorder=11)
ax1.add_feature(feature.LAND, facecolor='#cccccc', zorder=1)

# Time series
dict_fgco2['climatology_flux_PgC'].plot(ax=ax2, marker='o', color='b', lw=2.5, ms=7, label='Climatology (2000-2016)')
dict_fgco2['monthly_flux_PgC'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.5, ms=3, alpha=0.5, label='Monthly (2000-2016)')
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Pg C month$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (Pg C month$^{-1}$)')
ax2.legend(frameon=False)
ax3.legend(frameon=False)

fig.suptitle('Southern Ocean Sea Ice Biome', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


In [ ]:
# PLOTTING
map_da = (1000/365)*so_ice_fgco2.mean('time', skipna=True, keep_attrs=True).sel(lat=slice(-90, -35))

fig = plt.figure(figsize=[20, 5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_da.plot(ax=ax1, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='air-sea CO$_{2}$ (mmolC m$^{-2}$ day$^{-1}$)'), transform=crs.PlateCarree(), zorder=-1)
ax1.coastlines(lw=0.5, zorder=11)
ax1.add_feature(feature.LAND, facecolor='#cccccc', zorder=1)

# Time series
dict_fgco2['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='b', lw=2.5, ms=7, label='Climatology (2000-2016)')
dict_fgco2['monthly_flux_TgC_yr'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.5, ms=3, alpha=0.5, label='Monthly (2000-2016)')
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax2.legend(frameon=False)
ax3.legend(frameon=False)

fig.suptitle('Southern Ocean Sea Ice Biome', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


In [ ]:
# PLOTTING
map_da = (1000/365)*so_ice_fgco2.mean('time', skipna=True, keep_attrs=True).sel(lat=slice(-90, -35))

fig = plt.figure(figsize=[20, 5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_da.plot(ax=ax1, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='air-sea CO$_{2}$ (mmolC m$^{-2}$ day$^{-1}$)'), transform=crs.PlateCarree(), zorder=-1)
ax1.coastlines(lw=0.5, zorder=11)
ax1.add_feature(feature.LAND, facecolor='#cccccc', zorder=1)

# Time series
dict_fgco2['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='b', lw=2.5, ms=7, label='Climatology (2000-2016)')
dict_fgco2['monthly_flux_PgC_yr'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.5, ms=3, alpha=0.5, label='Monthly (2000-2016)')
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Pg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (Pg C yr$^{-1}$)')
ax2.legend(frameon=False)
ax3.legend(frameon=False)

fig.suptitle('Southern Ocean Sea Ice Biome', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


## Process air-sea CO$_2$ fluxes in the Sea Ice Biome sub-regions

In [ ]:
def map_ice_sectors(ax):
    clist = np.array(cm.tab20c.colors)
    new_colors = clist[[8, 1, 5,]].tolist()

    img = so_ice_sectors.plot.pcolormesh(
        ax=ax, transform=crs.PlateCarree(), 
        levels=np.array([0.5, 3.5, 6.5, 9.5]), 
        cbar_kwargs=dict(pad=0.03),
        colors=new_colors,
        rasterized=True,
        zorder=1)

    ax.coastlines(lw=0.5, zorder=11)
    ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=10)

    y = lambda lat: [lat] * x.size
    for idx, lat in enumerate([-33, -35, -46, -58]):
        deg_size = np.cos(np.deg2rad(lat)) ** 1.5
        n_degs = (10 / deg_size) / 2
        x = np.arange(0 + n_degs, 360 - n_degs, 1)
        
        if idx != 0:
            ax.plot(x, y(lat), transform=crs.PlateCarree(), color='k', lw=0.5, ls='--', alpha=0.4, zorder=2)
            ax.text(0, lat, f"{abs(lat)}°S", ha='center', va='center', transform=crs.PlateCarree(), size=8, alpha=0.4, zorder=2)
        else:
            ax.plot(x, y(lat), transform=crs.PlateCarree(), color='w', lw=0.05, ls='--', alpha=0.05, zorder=2)

    img.colorbar.set_ticks(np.array([2., 5., 8.]))
    img.colorbar.set_ticklabels(ice_names)
    img.colorbar.set_label('')
    [s.set_lw(0) for s in img.colorbar.ax.spines.values()]

    props = dict(ha='left', va='center', transform=crs.PlateCarree(), zorder=12, size=8, weight='light', fontfamily='helvetica')
    ax.text(149, -35, '147°E', **props)
    ax.text(20, -32, '20°E', **props)
    ax.text(292, -26, '70°W', **props)
    
    ax.spines['geo'].set_zorder(12)
    img.colorbar.ax.tick_params('both', length=0)


In [ ]:
fig = plt.figure(figsize=[7.5, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
map_ice_sectors(ax)
ax.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))


In [ ]:
# # Extract data for Pacific ICE
# avg = so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)

# #
# fig = plt.figure(figsize=[5, 5], dpi=300)
# ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))

# # Map
# avg.plot(ax=ax, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='air-sea CO$_{2}$ (mmolC m$^{-2}$ day$^{-1}$)'), transform=crs.PlateCarree(), zorder=-1)
# ax.coastlines(lw=0.5, zorder=11)
# ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=1)


In [ ]:
# Map
def sector_flux_map(da, ax):
    mol_to_mmol_day = 1000 / 365.0 # Convert from molC/m²/yr to mmolC/m²/day
    da = da * mol_to_mmol_day
    da.plot(ax=ax, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='air-sea CO$_{2}$ (mmol m$^{-2}$ day$^{-1}$)', extend='both'), transform=crs.PlateCarree(), zorder=-1)
    ax.coastlines(lw=0.5, zorder=11)
    ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=1)


In [ ]:
# Extract data for Pacific ICE
avg_atl = so_ice_fgco2.where(so_ice_sectors==3).mean('time', skipna=True, keep_attrs=True)
avg_ind = so_ice_fgco2.where(so_ice_sectors==6).mean('time', skipna=True, keep_attrs=True)
avg_pac = so_ice_fgco2.where(so_ice_sectors==9).mean('time', skipna=True, keep_attrs=True)
#
fig = plt.figure(figsize=[12, 4], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))
ax2 = fig.add_subplot(132, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))
ax3 = fig.add_subplot(133, projection=crs.Stereographic(central_latitude=-90, central_longitude=0, scale_factor=8))
sector_flux_map(avg_atl, ax1)
sector_flux_map(avg_ind, ax2)
sector_flux_map(avg_pac, ax3)
ax1.set_title('Atlantic ICE', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('Indian ICE', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('Pacific ICE', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))

fig.suptitle('Southern Ocean Sea Ice Biome - Subregions', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


In [ ]:
# # Extract data for Pacific ICE
# mask_atl = so_ice_sectors==3
# mask_ind = so_ice_sectors==6
# mask_pac = so_ice_sectors==9
# dict_fgco2_atl = process_airsea_flux(so_ice_fgco2)
# dict_fgco2_ind = process_airsea_flux(so_ice_fgco2)
# dict_fgco2_pac = process_airsea_flux(so_ice_fgco2)



In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
labels = ['Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2_atl['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, label='Atlantic ICE')
dict_fgco2_ind['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, label='Indian ICE')
dict_fgco2_pac['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, label='Pacific ICE')
#
dict_fgco2_atl['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='tab:green', lw=1.5, ms=3, alpha=0.5, label='Atlantic ICE')
dict_fgco2_ind['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='tab:blue', lw=1.5, ms=3, alpha=0.5, label='Indian ICE')
dict_fgco2_pac['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.5, ms=3, alpha=0.5, label='Pacific ICE')
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C month$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (Tg C month$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nMonthly (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('Southern Ocean Sea Ice Biome', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
labels = ['Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2_atl['climatology_flux_PgC'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, label='Atlantic ICE')
dict_fgco2_ind['climatology_flux_PgC'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, label='Indian ICE')
dict_fgco2_pac['climatology_flux_PgC'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, label='Pacific ICE')
#
dict_fgco2_atl['monthly_flux_PgC'].plot(ax=ax3, marker='o', color='tab:green', lw=1.5, ms=3, alpha=0.5, label='Atlantic ICE')
dict_fgco2_ind['monthly_flux_PgC'].plot(ax=ax3, marker='o', color='tab:blue', lw=1.5, ms=3, alpha=0.5, label='Indian ICE')
dict_fgco2_pac['monthly_flux_PgC'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.5, ms=3, alpha=0.5, label='Pacific ICE')
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Pg C month$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nMonthly (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('Southern Ocean Sea Ice Biome', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
labels = ['Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2_atl['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, label='Atlantic ICE')
dict_fgco2_ind['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, label='Indian ICE')
dict_fgco2_pac['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, label='Pacific ICE')
#
dict_fgco2_atl['monthly_flux_PgC_yr'].plot(ax=ax3, marker='o', color='tab:green', lw=1.5, ms=3, alpha=0.5, label='Atlantic ICE')
dict_fgco2_ind['monthly_flux_PgC_yr'].plot(ax=ax3, marker='o', color='tab:blue', lw=1.5, ms=3, alpha=0.5, label='Indian ICE')
dict_fgco2_pac['monthly_flux_PgC_yr'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.5, ms=3, alpha=0.5, label='Pacific ICE')
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Pg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (Pg C yr$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nMonthly (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('Southern Ocean Sea Ice Biome', fontsize=16, weight='bold', y=1.0)
fig.tight_layout(pad=1.5);


In [ ]:
def add_grid_ice_budget(ax, df=None):
    if df is not None:
        ax.xaxis.set_ticks(np.arange(len(df.index)))
        ax.set_xticklabels(df.index)
        ax.set_xlabel('')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2['climatology_flux_TgC'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7)
dict_fgco2_atl['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7)
dict_fgco2_ind['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7)
dict_fgco2_pac['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7)
#
dict_fgco2['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='k', lw=1.15, ms=2, alpha=0.5)
dict_fgco2_atl['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='tab:green', lw=1.15, ms=2, alpha=0.5)
dict_fgco2_ind['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='tab:blue', lw=1.15, ms=2, alpha=0.5)
dict_fgco2_pac['monthly_flux_TgC'].plot(ax=ax3, marker='o', color='tab:orange', lw=1.15, ms=2, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C month$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (Tg C month$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df)
add_grid_ice_budget(ax3)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nMonthly (2000 - 2016)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_TgC'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
dict_fgco2['climatology_flux_mmol_m2_day'].plot(ax=ax3, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_mmol_m2_day'].plot(ax=ax3, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_ind['climatology_flux_mmol_m2_day'].plot(ax=ax3, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_pac['climatology_flux_mmol_m2_day'].plot(ax=ax3, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C month$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (mmol m$^{-2}$ day$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Tg C month$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): mmol m$^{-2}$ day$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


In [ ]:
dict_fgco2 = process_airsea_flux(so_ice_fgco2)
list(dict_fgco2.keys())


In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
dict_fgco2['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_ind['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_pac['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (mmol m$^{-2}$ yr$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Tg C yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): mmol m$^{-2}$ yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_PgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
dict_fgco2['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_ind['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_pac['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Pg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (mmol m$^{-2}$ yr$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Pg C yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): mmol m$^{-2}$ yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


## Store to the disk

In [ ]:
# Compute sector fluxes
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
#
dict_csir_so = process_airsea_flux(da_so)
dict_csir_atl = process_airsea_flux(da_atl)
dict_csir_ind = process_airsea_flux(da_ind)
dict_csir_pac = process_airsea_flux(da_pac)


In [ ]:
import pandas as pd
import os

def save_df_to_excel(
    df: pd.DataFrame,
    file_path: str,
    sheet_name: str = "Sheet1",
    mode: str = "w",
    index: bool = False,
    engine: str = "openpyxl",
    **kwargs
):
    """
    Efficiently save a pandas DataFrame to an Excel file.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to save.
    file_path : str
        Full path to the Excel file (e.g., './outputs/data.xlsx').
    sheet_name : str, optional
        Sheet name to write to.
    mode : {'w', 'a'}, optional
        Write mode: 'w' to overwrite, 'a' to append to existing workbook.
    index : bool, optional
        Whether to include DataFrame index in the Excel sheet.
    engine : {'openpyxl', 'xlsxwriter'}, optional
        Engine to use for writing Excel files.
    **kwargs : dict
        Additional arguments passed to `DataFrame.to_excel()`.
    """
    # Ensure output directory exists
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    # Write efficiently depending on mode
    if mode == "a" and os.path.exists(file_path):
        with pd.ExcelWriter(file_path, engine=engine, mode=mode, if_sheet_exists="replace") as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=index, **kwargs)
    else:
        with pd.ExcelWriter(file_path, engine=engine, mode="w") as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=index, **kwargs)

    print(f"✅ Saved: {file_path} → sheet='{sheet_name}' ({len(df)} rows)")



In [ ]:
df = dict_csir_so['climatology_flux_mmol_m2_day'].reset_index()
df


In [ ]:
# # Save climatology to Excel
# df = dict_csir_so['climatology_flux_mmol_m2_day'].reset_index()
# file_path = '../data/Southern_Ocean_Sea_Ice_Biome_FCO2_mmol_m2_day.xlsx'
# save_df_to_excel(df, file_path, sheet_name='SO Sea Ice Biome (ICE)', index=False, mode='w', float_format="%.2f")
# #
# df = dict_csir_atl['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Atlantic ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_csir_ind['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Indian ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_csir_pac['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Pacific ICE', index=False, mode='a', float_format="%.2f")


In [ ]:
# # Save climatology to Excel
# df = dict_fgco2['climatology_flux_TgC'].reset_index()
# file_path = '../data/Southern_Ocean_Sea_Ice_Biome_FCO2_TgC_month.xlsx'
# save_df_to_excel(df, file_path, sheet_name='SO Sea Ice Biome (ICE)', index=False, mode='w', float_format="%.2f")
# #
# df = dict_fgco2_atl['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Atlantic ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_fgco2_ind['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Indian ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_fgco2_pac['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Pacific ICE', index=False, mode='a', float_format="%.2f")



In [ ]:
da_so


# MPI SOM-FFN V2023

## Load the data

In [ ]:
file_path = '../../../Collaborations/MPI_SOM-FFN_v2023_NCEI_OCADS.nc'
ds = xr.open_dataset(file_path)
mpi_ = ds.transpose('time', 'lat', 'lon')
mpi_


In [ ]:
# Create time coordinate (15th of each month)
start_year = 1982
end_year = 2022
time_mpi = pd.date_range(start=f"{start_year}-01-01",
                         end=f"{end_year}-12-01",
                         freq='MS'  # Month-start frequency
                        ) + pd.Timedelta(days=14)  # Shift to 15th of each month
mpi_ds = mpi_.assign_coords(time=time_mpi)
mpi_ds


In [ ]:
%%time
###
avgs_csir_ts = {}
da = mpi_ds.fgco2_raw.copy()
avgs_csir_ts["fgCO2_RAW"] = da.where(da!=1e20, np.nan).mean(["lat", "lon"], skipna=True, keep_attrs=True)
#
da = mpi_ds.fgco2_smoothed.copy()
avgs_csir_ts["fgCO2_Smoothed"] = da.where(da!=1e20, np.nan).mean(["lat", "lon"], skipna=True, keep_attrs=True)
#
da = mpi_ds.spco2_raw.copy()
avgs_csir_ts["spCO2_RAW"] = da.where(da!=1e20, np.nan).mean(["lat", "lon"], skipna=True, keep_attrs=True)
#
da = mpi_ds.spco2_smoothed.copy()
avgs_csir_ts["spCO2_Smoothed"] = da.where(da!=1e20, np.nan).mean(["lat", "lon"], skipna=True, keep_attrs=True)


In [ ]:
%%time
##### MPI-v2023
fig = plt.figure(figsize=[18,13], dpi=100)
axes = []
axes.append(fig.add_subplot(221))
axes.append(fig.add_subplot(222))
axes.append(fig.add_subplot(223))
axes.append(fig.add_subplot(224))

cbar_kwargs = {"orientation": "horizontal", "pad": 0.03, "shrink": 0.45, "extend": "max"}
###
for ax, key in zip(axes, avgs_csir_ts.keys()):
    avgs_csir_ts[key].plot(ax=ax)
    ax.set_title(key)
    
# ###
# for ax, key in zip(axes[2:], avgs_ts.keys()):
#     avgs_ts[key].plot(ax=ax)
    
fig.suptitle("\nMPI-SOMFFN-v2023 air-sea CO$_2$ time series (1982 - 2022)\n", fontsize=20)
fig.tight_layout();


## Process air-sea CO$_2$ fluxes in the Sea Ice Biome

In [ ]:
ds = mpi_ds[['fgco2_smoothed']].rename({'fgco2_smoothed': 'fgco2'}).sel(time=slice('2000-01-01', '2016-12-31'))

# Convert longitude
mpi_fgco2 = ds.fgco2.assign_coords(lon=((ds.lon + 360) % 360)).copy()
mpi_fgco2 = mpi_fgco2.sortby('lon')
mpi_fgco2 = mpi_fgco2.where(mpi_fgco2!=1e20, np.nan)
mpi_so_ice_fgco2 = mpi_fgco2.where(so_ice_mask==3).sel(lat=slice(-90, -35))
mpi_so_ice_fgco2


In [ ]:
da1 = so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)
da2 = mpi_so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)

fig = plt.figure(figsize=[15, 4], dpi=200)
ax1 = fig.add_subplot(121, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(122, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))

da1.plot(ax=ax1, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='F$_{CO2}$ (molC m$^{-2}$ yr$^{-1}$)'), transform=crs.PlateCarree())
da2.plot(ax=ax2, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='F$_{CO2}$ (molC m$^{-2}$ yr$^{-1}$)'), transform=crs.PlateCarree())

ax1.coastlines(lw=0.5, zorder=11)
ax1.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)
ax2.coastlines(lw=0.5, zorder=11)
ax2.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)
fig.tight_layout(pad=0.15);


## Process air-sea CO$_2$ fluxes in the Sea Ice Biome sub-regions

In [ ]:
# Extract data for Pacific ICE
da_so = so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
dict_fgco2['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_ind['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_pac['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (mmol m$^{-2}$ yr$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Tg C yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): mmol m$^{-2}$ yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nCSIR-ML6 fCO$_2$ Products\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


In [ ]:
# Extract data for Pacific ICE
da_so = mpi_so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
dict_fgco2['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_ind['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_pac['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (mmol m$^{-2}$ yr$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Tg C yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): mmol m$^{-2}$ yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nMPI SOM-FFN fCO$_2$ Products\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


## Store to the disk

In [ ]:
# Compute sector fluxes
da_so = mpi_so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
#
dict_mpi_so = process_airsea_flux(da_so)
dict_mpi_atl = process_airsea_flux(da_atl)
dict_mpi_ind = process_airsea_flux(da_ind)
dict_mpi_pac = process_airsea_flux(da_pac)
## Store to the disk


In [ ]:
# # Save climatology to Excel
# df = dict_mpi_so['climatology_flux_mmol_m2_day'].reset_index()
# file_path = '../data/MPI_Southern_Ocean_Sea_Ice_Biome_FCO2_mmol_m2_day.xlsx'
# save_df_to_excel(df, file_path, sheet_name='SO Sea Ice Biome (ICE)', index=False, mode='w', float_format="%.2f")
# #
# df = dict_mpi_atl['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Atlantic ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_mpi_ind['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Indian ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_mpi_pac['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Pacific ICE', index=False, mode='a', float_format="%.2f")


In [ ]:
# # Save climatology to Excel
# df = dict_mpi_so['climatology_flux_TgC'].reset_index()
# file_path = '../data/MPI_Southern_Ocean_Sea_Ice_Biome_FCO2_TgC_month.xlsx'
# save_df_to_excel(df, file_path, sheet_name='SO Sea Ice Biome (ICE)', index=False, mode='w', float_format="%.2f")
# #
# df = dict_mpi_atl['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Atlantic ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_mpi_ind['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Indian ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_mpi_pac['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Pacific ICE', index=False, mode='a', float_format="%.2f")



In [ ]:
df = dict_mpi_so['climatology_flux_TgC'].copy()
df.plot(kind='bar')
df.sum()


In [ ]:
df = dict_csir_so['climatology_flux_TgC'].copy()
df.plot(kind='bar')
df.sum()


In [ ]:
df = dict_mpi_so['climatology_flux_PgC_yr'].copy()
df.plot(kind='bar')
df.sum()


In [ ]:
df = dict_csir_so['climatology_flux_PgC_yr'].copy()
df.plot(kind='bar')
df.sum()


In [ ]:
df = dict_csir_so['climatology_flux_TgC'].copy()
df.plot(kind='bar')
df.sum()


# Ensemble mean of air-sea CO$_2$ fluxes


Here we are averaging both CSIR-ML6 & MPI-SOMFFN fCO$_2$ Products to create a robust one

## Processing CO$_2$ fluxes in the Sea Ice Biome

In [ ]:
# Extract data from both datasets
ds1 = csirml6v2019[['FCO2_smooth']].rename({'FCO2_smooth': 'fgco2'}).sel(time=slice('2000-01-01', '2016-12-31'))
ds2 = mpi_ds[['fgco2_smoothed']].rename({'fgco2_smoothed': 'fgco2'}).sel(time=slice('2000-01-01', '2016-12-31'))

# Convert longitude
fgco2_ds1 = ds1.assign_coords(lon=((ds1.lon + 360) % 360)).copy()
fgco2_ds2 = ds2.assign_coords(lon=((ds2.lon + 360) % 360)).copy()
fgco2_ds1 = fgco2_ds1.sortby('lon')
fgco2_ds2 = fgco2_ds2.sortby('lon')

# Filter missing values on MPI-SOMFFN Product
fgco2_ds2 = fgco2_ds2.where(fgco2_ds2!=1e20, np.nan)

# Create the ensemble dictionary
fgco2_dict = {'CSIR-ML6': fgco2_ds1,
              'MPI-SOMFFN': fgco2_ds2}


In [ ]:
fgco2_ds2


In [ ]:
# Create FCO2 ensemble

# List of FCO2 DataArrays
fgco2_list = []
productcs = []
for key in fgco2_dict.keys():
    productcs += key,
    print(productcs[-1], end="\t")
    
    da = fgco2_dict[key].fgco2.copy()
    fgco2_list.append(da)

# Stack all FCO2 DataArrays along a new dimension
fgco2_ds = xr.concat(fgco2_list, dim='new_dim')

# Calculate the mean while ignoring NaNs
fgco2_ens = fgco2_ds.mean(dim='new_dim', skipna=True)


# Masking: if a grid cell is NaN across all DataArrays, it remains NaN in the mean
fgco2_ens = fgco2_ens.where(~fgco2_ds.isnull().all(dim='new_dim'))
fgco2_ens.attrs = {"long_name": "air-sea CO2 fluxes", "units": "mol m-2 yr-1"}

### Merge
fgco2_ens = fgco2_ens.to_dataset(name='fgco2')
fgco2_ens.attrs = {"description": ("Ensemble mean air-sea CO2 fluxes from CSIR-ML6 and MPI-SOMFFN products"),
                   "created_by": "Laique M. Djeutchouang",
                   "contact": "merlindjeutchouang@gmail.com",
                   "creation_date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}

# Extract Southern Ocean Sea Ice Biome
ens_so_ice_fgco2 = fgco2_ens.fgco2.where(so_ice_mask==3).sel(lat=slice(-90, -35))

fgco2_ens


In [ ]:
da1 = so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)
da2 = mpi_so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)
da3 = ens_so_ice_fgco2.mean('time', skipna=True, keep_attrs=True)

fig = plt.figure(figsize=[15, 4], dpi=200)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax3 = fig.add_subplot(133, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))

da1.plot(ax=ax1, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='F$_{CO2}$ (mol m$^{-2}$ yr$^{-1}$)'), transform=crs.PlateCarree())
da2.plot(ax=ax2, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='F$_{CO2}$ (mol m$^{-2}$ yr$^{-1}$)'), transform=crs.PlateCarree())
da3.plot(ax=ax3, robust=True, cmap='RdBu_r', cbar_kwargs=dict(label='F$_{CO2}$ (mol m$^{-2}$ yr$^{-1}$)'), transform=crs.PlateCarree())

ax1.coastlines(lw=0.5, zorder=11)
ax1.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)
ax2.coastlines(lw=0.5, zorder=11)
ax2.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)
ax3.coastlines(lw=0.5, zorder=11)
ax3.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)
ax1.set_title('CSIR-ML6 fCO$_2$ Product', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('MPI SOM-FFN fCO$_2$ Product', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('Ensemble Mean fCO$_2$ Product', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.tight_layout(pad=0.15);


## Processing CO$_2$ fluxes in the Sea Ice Biome sub-regions

In [ ]:
# Extract data for Pacific ICE
da_so = ens_so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors(ax1)

# Time series
dict_fgco2['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
dict_fgco2['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_ind['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_pac['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (mmol m$^{-2}$ yr$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Tg C yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): mmol m$^{-2}$ yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nEnsemble Mean Air-Sea CO$_2$ Fluxes from CSIR-ML6 & MPI-SOMFFN fCO$2$ Products\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


## Store to the disk

In [ ]:
# Compute sector fluxes
da_so = ens_so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
#
dict_ens_so = process_airsea_flux(da_so)
dict_ens_atl = process_airsea_flux(da_atl)
dict_ens_ind = process_airsea_flux(da_ind)
dict_ens_pac = process_airsea_flux(da_pac)



In [ ]:
# # Save climatology to Excel
# df = dict_ens_so['climatology_flux_mmol_m2_day'].reset_index()
# file_path = '../data/ENSEMBLE_Southern_Ocean_Sea_Ice_Biome_FCO2_mmol_m2_day.xlsx'
# save_df_to_excel(df, file_path, sheet_name='SO Sea Ice Biome (ICE)', index=False, mode='w', float_format="%.2f")
# #
# df = dict_ens_atl['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Atlantic ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_ens_ind['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Indian ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_ens_pac['climatology_flux_mmol_m2_day'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Pacific ICE', index=False, mode='a', float_format="%.2f")



In [ ]:
# # Save climatology to Excel
# df = dict_ens_so['climatology_flux_TgC'].reset_index()
# file_path = '../data/ENSEMBLE_Southern_Ocean_Sea_Ice_Biome_FCO2_TgC_month.xlsx'
# save_df_to_excel(df, file_path, sheet_name='SO Sea Ice Biome (ICE)', index=False, mode='w', float_format="%.2f")
# #
# df = dict_ens_atl['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Atlantic ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_ens_ind['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Indian ICE', index=False, mode='a', float_format="%.2f")
# #
# df = dict_ens_pac['climatology_flux_TgC'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Pacific ICE', index=False, mode='a', float_format="%.2f")



In [ ]:
df = dict_csir_so['climatology_flux_TgC'].copy()
df.plot(kind='bar')
df.sum()


In [ ]:
df = dict_mpi_so['climatology_flux_TgC'].copy()
df.plot(kind='bar')
df.sum()


In [ ]:
df = dict_ens_so['climatology_flux_TgC'].copy()
df.plot(kind='bar')
df.sum()


In [ ]:
list(dict_ens_so.keys())


In [ ]:
df = dict_ens_so['climatology_flux_mmol_m2_day'].copy()
df.plot(kind='bar')
df.sum()


# Area of each region in km2

Check the monthly sea ice area and the monthly air-sea fluxes in mm m2 day of the assembly.

## Tools

In [ ]:
import pyseaflux as psf
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']


In [ ]:
def get_ocean_area_km2(da, var_name=''):
    ### Calculate the overall area 
    area_m2 = psf.area.get_area_from_dataset(da) # this is in m2
    area_km2 = area_m2 * 1e-6 # convert to km2: m2 --> km2 = m2*1e-6 (m2/km2)
    
    ### Get the flux-based seamask and derive the flux area
    # flx_mask = 1*(da.mean("time", skipna=True)**2 >= 0.0)
    # area = area_m2.where(flx_mask==1)
    area = area_km2.copy()
    area.name = f"area_{var_name}"
    area.attrs = area_m2.attrs.copy()
    area.attrs["units"] = "km2"
    
    return area.to_dataset()


## Compute area of each ocean region of interest

In [ ]:
ds = get_ocean_area_km2(da=so_ice_sectors, var_name='')
ds


In [ ]:
# Compute area
da = ds.area_.copy()

# Plot area
fig = plt.figure(figsize=[4, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
da.plot(ax=ax, robust=True, transform=crs.PlateCarree())
ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=11)

print(f"Total area of the Southern Ocean (< 35ºS): {da.sum(['lat', 'lon']).values:.2f} km2")  # in km2


In [ ]:
# Compute area
da = ds.area_.where(so_ice_sectors==3)
sector = labels[1]

# Plot area
fig = plt.figure(figsize=[4, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
da.plot(ax=ax, robust=True, transform=crs.PlateCarree())
ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)

print(f"Total area of the {sector}: {da.sum(['lat', 'lon']).values:.2f} km2")  # in km2


In [ ]:
# Compute area
da = ds.area_.where(so_ice_sectors==6)
sector = labels[2]

# Plot area
fig = plt.figure(figsize=[4, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
da.plot(ax=ax, robust=True, transform=crs.PlateCarree())
ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)

print(f"Total area of the {sector}: {da.sum(['lat', 'lon']).values:.2f} km2")  # in km2


In [ ]:
# Compute area
da = ds.area_.where(so_ice_sectors==9)
sector = labels[3]

# Plot area
fig = plt.figure(figsize=[4, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
da.plot(ax=ax, robust=True, transform=crs.PlateCarree())
ax.coastlines(lw=0.5, zorder=11)
ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=-1)

print(f"Total area of the {sector}: {da.sum(['lat', 'lon']).values:.2f} km2")  # in km2


In [ ]:
ice_names



In [ ]:
def map_ice_sectors_(ax):
    clist = np.array(cm.tab20c.colors)
    new_colors = clist[[8, 1, 5,]].tolist()

    img = so_ice_sectors.plot.pcolormesh(
        ax=ax, transform=crs.PlateCarree(), 
        levels=np.array([0.5, 3.5, 6.5, 9.5]), 
        cbar_kwargs=dict(pad=0.03),
        colors=new_colors,
        rasterized=True,
        zorder=1)

    ax.coastlines(lw=0.5, zorder=11)
    ax.add_feature(feature.LAND, facecolor='#cccccc', zorder=10)

    y = lambda lat: [lat] * x.size
    for idx, lat in enumerate([-33, -35, -46, -58]):
        deg_size = np.cos(np.deg2rad(lat)) ** 1.5
        n_degs = (10 / deg_size) / 2
        x = np.arange(0 + n_degs, 360 - n_degs, 1)
        
        if idx != 0:
            ax.plot(x, y(lat), transform=crs.PlateCarree(), color='k', lw=0.5, ls='--', alpha=0.4, zorder=2)
            ax.text(0, lat, f"{abs(lat)}°S", ha='center', va='center', transform=crs.PlateCarree(), size=8, alpha=0.4, zorder=2)
        else:
            ax.plot(x, y(lat), transform=crs.PlateCarree(), color='w', lw=0.05, ls='--', alpha=0.05, zorder=2)
    # Customize colorbar
    area_ = get_ocean_area_km2(so_ice_sectors).area_.copy()
    ice_names_ = [f"Atlantic ICE\n({area_.where(so_ice_sectors==3).sum(['lat', 'lon']).values:.2f} km$^2$)",
                  f"Indian ICE\n({area_.where(so_ice_sectors==6).sum(['lat', 'lon']).values:.2f} km$^2$)",
                  f"Pacific ICE\n({area_.where(so_ice_sectors==9).sum(['lat', 'lon']).values:.2f} km$^2$)"]
    img.colorbar.set_ticks(np.array([2., 5., 8.]))
    img.colorbar.set_ticklabels(ice_names_, fontsize=8, va="center", ha="center")
    img.colorbar.set_label('')
    [s.set_lw(0) for s in img.colorbar.ax.spines.values()]
    img.colorbar.ax.yaxis.set_label_coords(1.25, 0.5)
    img.colorbar.ax.yaxis.set_tick_params(labelleft=False, labelright=True, pad=40)

    props = dict(ha='left', va='center', transform=crs.PlateCarree(), zorder=12, size=8, weight='light', fontfamily='helvetica')
    ax.text(149, -35, '147°E', **props)
    ax.text(20, -32, '20°E', **props)
    ax.text(292, -26, '70°W', **props)
    
    ax.spines['geo'].set_zorder(12)
    img.colorbar.ax.tick_params('both', length=0)


In [ ]:
# Plot ice sectors
fig = plt.figure(figsize=[7.5, 4], dpi=200)
ax = fig.add_subplot(111, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
map_ice_sectors_(ax)
ax.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))


In [ ]:
# Extract data for Pacific ICE
da_so = ens_so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors_(ax1)

# Time series
dict_fgco2['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
dict_fgco2['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_ind['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.5)
dict_fgco2_pac['climatology_flux_mmol_m2_year'].plot(ax=ax3, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.5)
#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax3.set_ylabel('F$_{CO2}$ (mmol m$^{-2}$ yr$^{-1}$)')
ax2.legend(labels=labels, frameon=False)
ax3.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Tg C yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): mmol m$^{-2}$ yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nEnsemble Mean Air-Sea CO$_2$ Fluxes from CSIR-ML6 & MPI-SOMFFN fCO$2$ Products\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


In [ ]:
fice = mpi_ds.ice.sel(time=slice('2000-01-01', '2016-12-31')).copy()

# Convert longitude
fice = fice.assign_coords(lon=((ds.lon + 360) % 360)).copy()
fice = fice.sortby('lon')
fice = fice.where(fice!=1e20, np.nan)
fice_so = fice.sel(lat=slice(-90, -34))
clim_fice_so = fice_so.groupby('time.month').mean('time', skipna=True, keep_attrs=True)
# fice_so = fice.where(so_ice_mask==3).sel(lat=slice(-90, -35))
fice_so


In [ ]:
fice_so[0].plot()


In [ ]:
clim_fice_so[0].plot()


In [ ]:
da_ = clim_fice_so.copy()
for i in range(12):
    da_[i].values = ds.area_.values
    
da_


In [ ]:
ds.area_.plot()


In [ ]:
clim_fice_so[8].where(clim_fice_so[8]>0).plot()


In [ ]:
fice_so[8].plot()


In [ ]:
def process_sea_ice_biomes_area(da):
    """
    Compute monthly mean air-sea CO2 fluxes and total carbon fluxes from a
    DataArray of molC/m²/yr, including climatological and time-series values.

    Parameters
    ----------
    flux : xr.DataArray
        Ice coverage [%] with dimensions (time, lat, lon)
    mask : xr.DataArray, optional

    Returns
    -------
    results : dict
    """
    # Compute area per grid cell
    area_ = get_ocean_area_km2(da).area_.copy()
    
    # --- STEP 2: Compute climatological monthly means ---
    clim = da.groupby("time.month").mean('time', skipna=True, keep_attrs=True)
    
    # --- STEP 3: Apply regional mask ---
    seaice_area = {"SO_Sea_Ice_Region": ["Atlantic ICE", "Indian ICE", "Pacific ICE"],
                   "Sea_Ice_Area_km2": [np.float64(area_.where(so_ice_sectors==3).sum(['lat', 'lon']).values),
                                    np.float64(area_.where(so_ice_sectors==6).sum(['lat', 'lon']).values),
                                    np.float64(area_.where(so_ice_sectors==9).sum(['lat', 'lon']).values)]}
    
    seaice_area = pd.DataFrame(seaice_area)
    
    return seaice_area


In [ ]:
seaice_area = process_sea_ice_biomes_area(fice_so)
seaice_area


In [ ]:
def process_seaice_area(da):
    """
    Compute monthly mean air-sea CO2 fluxes and total carbon fluxes from a
    DataArray of molC/m²/yr, including climatological and time-series values.

    Parameters
    ----------
    flux : xr.DataArray
        Ice coverage [%] with dimensions (time, lat, lon)
    mask : xr.DataArray, optional

    Returns
    -------
    results : dict
    """

    # --- CONSTANTS ---
    index = da.time.dt.month.to_dataframe().drop_duplicates().index.strftime('%b')

    # Compute area per grid cell
    area_ = get_ocean_area_km2(so_ice_sectors).area_.copy()
    
    # --- STEP 2: Compute climatological monthly means ---
    clim = da.groupby("time.month").mean('time', skipna=True, keep_attrs=True)
    
    # --- STEP 3: Apply regional mask ---
    monthly_seaice_area = [np.float64(area_.where(clim[i]>0).sum(['lat', 'lon']).values) for i in range(12)]
    
    monthly_seaice_area = pd.DataFrame({'time': index,
                                        'Sea_Ice_Area_km2': monthly_seaice_area})
    monthly_seaice_area = monthly_seaice_area.set_index('time')
    
    return monthly_seaice_area


In [ ]:
monthly_seaice_area = process_seaice_area(fice_so)
monthly_seaice_area


In [ ]:
# Extract data for Pacific ICE
da_so = ens_so_ice_fgco2.sel(lat=slice(-90, -35))
da_atl = da_so.where(so_ice_sectors==3)
da_ind = da_so.where(so_ice_sectors==6)
da_pac = da_so.where(so_ice_sectors==9)
dict_fgco2_atl = process_airsea_flux(da_atl)
dict_fgco2_ind = process_airsea_flux(da_ind)
dict_fgco2_pac = process_airsea_flux(da_pac)
df = dict_fgco2_pac['climatology_flux_mmol_m2_day'].copy()
labels = ['Sea Ice Biome', 'Atlantic ICE', 'Indian ICE', 'Pacific ICE']
#
monthly_ice_area = process_seaice_area(fice_so)

# PLOTTING
fig = plt.figure(figsize=[20, 5.5], dpi=300)
ax1 = fig.add_subplot(131, projection=crs.Stereographic(central_latitude=-90, central_longitude=0))
ax2 = fig.add_subplot(132)
ax3 = fig.add_subplot(133)

# Map
map_ice_sectors_(ax1)

# Time series
dict_fgco2['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='k', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_atl['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:green', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_ind['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:blue', lw=2.5, ms=7, alpha=0.75)
dict_fgco2_pac['climatology_flux_TgC_yr'].plot(ax=ax2, marker='o', color='tab:orange', lw=2.5, ms=7, alpha=0.75)
#
monthly_ice_area['Sea_Ice_Area_km2'].plot(ax=ax3, marker='o', color='tab:red', lw=2.5, ms=7, alpha=0.75)

#
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax2.set_ylabel('F$_{CO2}$ (Tg C yr$^{-1}$)')
ax3.set_ylabel('Ice Coverage Area (km$^{-2}$)')
ax2.legend(labels=labels, frameon=False)
add_grid_ice_budget(ax2, df=df)
add_grid_ice_budget(ax3, df=df)

ax1.set_title('\nSouthern Ocean Sea Ice Biome (ICE)\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax2.set_title('\nClimatology (2000 - 2016): Tg C yr$^{-1}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
ax3.set_title('\nClimatology (2000 - 2016): km$^{-2}$\n', fontdict=dict(size=12, weight='bold', fontfamily='helvetica'))
fig.suptitle('\nEnsemble Mean Air-Sea CO$_2$ Fluxes from CSIR-ML6 & MPI-SOMFFN fCO$2$ Products\nSouthern Ocean Sea Ice Biome\n', fontsize=16, weight='bold', y=1.02)
fig.tight_layout(pad=.5);


## Store to the disk

In [ ]:
# # Save climatology to Excel
# monthly_ice_area = process_seaice_area(fice_so)
# df = monthly_ice_area['Sea_Ice_Area_km2'].reset_index()
# file_path = '../data/Southern_Ocean_monthly_sea_ice_area_and_regional_biomes.xlsx'
# save_df_to_excel(df, file_path, sheet_name='Monthly SO Sea Ice Area', index=False, mode='w', float_format="%.2f")
# #
# seaice_area = process_sea_ice_biomes_area(fice_so)
# df = seaice_area['Sea_Ice_Area_km2'].reset_index()
# save_df_to_excel(df, file_path, sheet_name='SO Sea Ice Biome (ICE) Area', index=False, mode='a', float_format="%.2f")
